In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path(r"C:\Users\cetin\OneDrive\MASAST~1\TASARIM\League of Legends Ranked Match Data  Season 15 (EUN).csv")
assert DATA_PATH.exists(), f"CSV not found: {DATA_PATH}"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)



In [ ]:
# Hızlı yükleme ve temel şekil

df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
print("columns:", df.columns.tolist())

# ilk satırlar
df.head()



In [ ]:
# Veri tipleri ve hızlı özet

info = df.dtypes.to_frame("dtype")
info["non_null"] = df.notna().sum()
info["na_ratio"] = 1 - info["non_null"] / len(df)
info.sort_values("na_ratio", ascending=False).head(15)



In [ ]:
# Eksik değer yüzdeleri (ilk 20)
na_pct = df.isna().mean().sort_values(ascending=False)
na_pct.head(20)



In [ ]:
# Sayısal kolon özetleri
numeric_cols = df.select_dtypes(include=[np.number]).columns
summary = df[numeric_cols].describe().T
summary.head(15)



In [ ]:
# Hedef dağılımları (win, rank örnekleri)
print(df['win'].value_counts(normalize=True).rename('win_ratio'))
if 'solo_tier' in df.columns:
    print(df['solo_tier'].value_counts().head())



In [ ]:
# Modelleme için temel sütun grupları

target_col = 'win'

id_cols = [
    'game_id', 'participant_id', 'champion_id', 'champion_name',
    'position', 'start_utc', 'queue', 'platform_id', 'map_id',
    'game_mode', 'game_version'
]

numeric_features = [
    'kills', 'deaths', 'assists', 'kda_ratio', 'kill_participation',
    'gold_earned', 'gold_spent', 'gold_per_min',
    'damage_dealt', 'damage_per_min', 'damage_to_champ', 'damage_champ_per_min',
    'damage_taken', 'vision_score',
    'final_abilityHaste', 'final_abilityPower', 'final_armor',
    'final_attackDamage', 'final_attackSpeed', 'final_movementSpeed',
    'final_health', 'final_healthMax', 'final_lifesteal', 'final_omnivamp',
    'final_power', 'final_powerMax'
]

cat_features = [
    'position', 'solo_tier', 'solo_rank', 'flex_tier', 'flex_rank', 'champion_name'
]

# Gerçekte var olanları filtreleyelim
id_cols = [c for c in id_cols if c in df.columns]
numeric_features = [c for c in numeric_features if c in df.columns]
cat_features = [c for c in cat_features if c in df.columns]

print('ID cols:', id_cols)
print('Numeric:', len(numeric_features))
print('Categorical:', len(cat_features))

# Hedef dağılımı
df[target_col].value_counts(normalize=True).rename('win_ratio')


In [ ]:
# Eksik değer analizi (modelde kullanılacak kolonlar için)

use_cols = numeric_features + cat_features + [target_col]
use_cols = [c for c in use_cols if c in df.columns]

na_pct_model = df[use_cols].isna().mean().sort_values(ascending=False)
na_pct_model.head(20)


In [ ]:
# Basit imputasyon (sayısal: median, kategorik: en sık)

from sklearn.impute import SimpleImputer

df_model = df.copy()

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

if numeric_features:
    df_num = pd.DataFrame(
        num_imputer.fit_transform(df_model[numeric_features]),
        columns=numeric_features,
        index=df_model.index,
    )
else:
    df_num = pd.DataFrame(index=df_model.index)

if cat_features:
    df_cat = pd.DataFrame(
        cat_imputer.fit_transform(df_model[cat_features]),
        columns=cat_features,
        index=df_model.index,
    )
else:
    df_cat = pd.DataFrame(index=df_model.index)

# İleride gerekirse kolayca yeniden birleştirmek için sadece ayrı DataFrame'ler oluşturuyoruz
df_num.head()


In [ ]:
# Aykırı değerleri anlamak için hızlı özet (IQR bakış)

import numpy as np

summary_num = df_num.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
summary_num[['mean', 'std', 'min', '1%', '25%', '50%', '75%', '99%', 'max']].head(15)


In [ ]:
# Örnek: birkaç ana metrikte uç değerleri kırpma (winsorizing)

clip_cols = [c for c in ['kills', 'deaths', 'assists', 'gold_earned', 'damage_to_champ'] if c in df_num.columns]

for col in clip_cols:
    q_low = df_num[col].quantile(0.01)
    q_high = df_num[col].quantile(0.99)
    df_num[col] = df_num[col].clip(q_low, q_high)

summary_clip = df_num[clip_cols].describe(percentiles=[0.01, 0.5, 0.99]).T
summary_clip


In [ ]:
# Ölçekleme ve train/test bölme için iskelet

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Kategorik değişkenleri one-hot encode edelim
if not df_cat.empty:
    X_cat = pd.get_dummies(df_cat, drop_first=True)
else:
    X_cat = pd.DataFrame(index=df_model.index)

X_num = df_num.copy()

X = pd.concat([X_num, X_cat], axis=1)

y = df_model[target_col].astype(int)  # win sütunu genelde 0/1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape


In [ ]:
# Feature Engineering: yeni oranlar ve verimlilik metrikleri

fe_df = df_model.copy()

# Güvenli bölme fonksiyonu
EPS = 1e-6

def safe_div(a, b):
    return a / (b.replace(0, np.nan) + EPS)

# 1) KDA ve benzeri (bazıları zaten varsa, yoksa üret)
if 'kills' in fe_df.columns and 'deaths' in fe_df.columns and 'assists' in fe_df.columns:
    fe_df['kda_calc'] = (fe_df['kills'] + fe_df['assists']) / (fe_df['deaths'].replace(0, np.nan) + EPS)

# 2) Altın ve hasar verimliliği
if {'gold_earned', 'damage_to_champ'} <= set(fe_df.columns):
    fe_df['dmg_per_gold'] = safe_div(fe_df['damage_to_champ'], fe_df['gold_earned'])

if {'gold_earned', 'kills'} <= set(fe_df.columns):
    fe_df['gold_per_kill'] = safe_div(fe_df['gold_earned'], fe_df['kills'])

# 3) Ölüm başına hasar ve skor
if {'damage_to_champ', 'deaths'} <= set(fe_df.columns):
    fe_df['dmg_per_death'] = safe_div(fe_df['damage_to_champ'], fe_df['deaths'])

if {'kills', 'deaths'} <= set(fe_df.columns):
    fe_df['kill_death_ratio'] = safe_div(fe_df['kills'], fe_df['deaths'])

# 4) Vizyon ve takım katkısı oranları
if {'vision_score', 'duration'} <= set(fe_df.columns):
    fe_df['vision_per_min'] = safe_div(fe_df['vision_score'], (fe_df['duration'] / 60.0))

if {'kill_participation', 'team_champKills'} <= set(fe_df.columns):
    # kill_participation genelde oran; yine de saklıyoruz
    fe_df['kp_x_teamkills'] = fe_df['kill_participation'] * fe_df['team_champKills']

# 5) Sağlık ve dayanıklılık oranları
if {'final_healthMax', 'damage_taken'} <= set(fe_df.columns):
    fe_df['dmg_taken_per_hp'] = safe_div(fe_df['damage_taken'], fe_df['final_healthMax'])

# Yeni üretilen kolonlar listesini görelim
new_fe_cols = [c for c in fe_df.columns if c not in df_model.columns]
new_fe_cols, fe_df[new_fe_cols].describe().T.head(10)


In [ ]:
# Feature Engineering sonrası: numeric_features listesine ekleme

fe_numeric_add = [
    c for c in [
        'kda_calc', 'dmg_per_gold', 'gold_per_kill', 'dmg_per_death',
        'kill_death_ratio', 'vision_per_min', 'kp_x_teamkills',
        'dmg_taken_per_hp'
    ]
    if c in fe_df.columns
]

numeric_features_fe = list(dict.fromkeys(numeric_features + fe_numeric_add))  # sıralı benzersiz

print('Yeni FE numeric:', fe_numeric_add)
print('Toplam numeric (FE ile):', len(numeric_features_fe))


In [ ]:
# FE sonrası imputasyon ve model matrisi (güncel akış)

# Yeni df_model: FE uygulanmış sürüm
df_model = fe_df.copy()

# Sayısal / kategorik aynı mantıkla, FE içeren numeric listesiyle yeniden oluşturuluyor
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

if numeric_features_fe:
    df_num = pd.DataFrame(
        num_imputer.fit_transform(df_model[numeric_features_fe]),
        columns=numeric_features_fe,
        index=df_model.index,
    )
else:
    df_num = pd.DataFrame(index=df_model.index)

if cat_features:
    df_cat = pd.DataFrame(
        cat_imputer.fit_transform(df_model[cat_features]),
        columns=cat_features,
        index=df_model.index,
    )
else:
    df_cat = pd.DataFrame(index=df_model.index)

# Kategorik one-hot + ölçekleme + train/test tekrar
if not df_cat.empty:
    X_cat = pd.get_dummies(df_cat, drop_first=True)
else:
    X_cat = pd.DataFrame(index=df_model.index)

X_num = df_num.copy()
X = pd.concat([X_num, X_cat], axis=1)

y = df_model[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape


In [ ]:
# Modelleme: Kazanma Tahmin Modelleri
# Logistic Regression, Random Forest, XGBoost

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, classification_report
)
import warnings
warnings.filterwarnings('ignore')

# Modelleri eğitme
models = {}

# 1. Logistic Regression
print("=" * 50)
print("Logistic Regression Eğitiliyor...")
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)
models['Logistic Regression'] = lr

# 2. Random Forest
print("Random Forest Eğitiliyor...")
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf.fit(X_train, y_train)  # RF için ölçeklenmemiş veri
models['Random Forest'] = rf

# 3. XGBoost (varsa)
try:
    import xgboost as xgb
    print("XGBoost Eğitiliyor...")
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    models['XGBoost'] = xgb_model
    print("XGBoost başarıyla eğitildi.")
except ImportError:
    print("XGBoost bulunamadı. 'pip install xgboost' ile yükleyebilirsiniz.")

print("\nTüm modeller eğitildi!")


In [ ]:
# Model Değerlendirme: Metrikler ve Karşılaştırma

results = {}

for name, model in models.items():
    # Tahminler
    if name == 'Logistic Regression':
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrikler
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'AUC-ROC': auc,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"\n{name}:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")

# Sonuçları DataFrame'e çevir
results_df = pd.DataFrame(results).T
results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']]


In [ ]:
# Confusion Matrix görselleştirme

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, len(models), figsize=(5*len(models), 4))
if len(models) == 1:
    axes = [axes]

for idx, (name, model) in enumerate(models.items()):
    y_pred = results[name]['y_pred']
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        ax=axes[idx], cbar=False
    )
    axes[idx].set_title(f'{name}\nConfusion Matrix')
    axes[idx].set_xlabel('Tahmin')
    axes[idx].set_ylabel('Gerçek')

plt.tight_layout()
plt.show()


In [ ]:
# ROC Curve görselleştirme

plt.figure(figsize=(10, 6))

for name in models.keys():
    y_pred_proba = results[name]['y_pred_proba']
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = results[name]['AUC-ROC']
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Karşılaştırması')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Oyuncu Persona Kümelemesi: K-Means

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Kümeleme için özellik seçimi (oyuncu performans metrikleri)
cluster_features = [
    'kills', 'deaths', 'assists', 'kda_ratio',
    'gold_earned', 'damage_to_champ', 'vision_score',
    'kill_participation'
]

# Sadece var olanları kullan
cluster_features = [c for c in cluster_features if c in X_num.columns]

if cluster_features:
    X_cluster = X_num[cluster_features].copy()
    
    # Ölçekleme
    scaler_cluster = StandardScaler()
    X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)
    
    # Optimal küme sayısını bulma (Elbow Method)
    inertias = []
    sil_scores = []
    K_range = range(2, 8)
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X_cluster_scaled)
        inertias.append(kmeans.inertia_)
        sil_scores.append(silhouette_score(X_cluster_scaled, kmeans.labels_))
    
    # En iyi k (silhouette score'a göre)
    best_k = K_range[np.argmax(sil_scores)]
    print(f"En iyi küme sayısı (Silhouette): {best_k}")
    print(f"Silhouette Score: {max(sil_scores):.4f}")
    
    # Final kümeleme
    kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    cluster_labels = kmeans_final.fit_predict(X_cluster_scaled)
    
    # Sonuçları DataFrame'e ekle
    df_cluster = X_cluster.copy()
    df_cluster['cluster'] = cluster_labels
    
    # Küme özetleri
    cluster_summary = df_cluster.groupby('cluster')[cluster_features].mean()
    print("\nKüme Özetleri (Ortalama Değerler):")
    print(cluster_summary)
    
else:
    print("Kümeleme için yeterli özellik bulunamadı.")


In [ ]:
# Kümeleme görselleştirme: PCA ile 2D/3D

if cluster_features and 'cluster' in df_cluster.columns:
    # PCA ile boyut indirgeme
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_cluster_scaled)
    
    # 2D scatter plot
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(
        X_pca[:, 0], X_pca[:, 1],
        c=cluster_labels, cmap='viridis', alpha=0.6, s=50
    )
    plt.colorbar(scatter, label='Küme')
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
    plt.title('Oyuncu Persona Kümelemesi (PCA ile 2D Görselleştirme)')
    plt.grid(alpha=0.3)
    plt.show()
    
    # Küme dağılımı
    plt.figure(figsize=(8, 5))
    df_cluster['cluster'].value_counts().sort_index().plot(kind='bar')
    plt.xlabel('Küme')
    plt.ylabel('Oyuncu Sayısı')
    plt.title('Küme Dağılımı')
    plt.xticks(rotation=0)
    plt.grid(alpha=0.3, axis='y')
    plt.show()


In [ ]:
# Açıklanabilir Yapay Zeka (XAI): SHAP Değerleri

try:
    import shap
    
    # En iyi modeli seç (AUC'ye göre)
    best_model_name = max(results.keys(), key=lambda k: results[k]['AUC-ROC'])
    best_model = models[best_model_name]
    
    print(f"SHAP analizi için kullanılan model: {best_model_name}")
    
    # SHAP için veri hazırlama
    if best_model_name == 'Logistic Regression':
        X_shap = X_test_scaled[:100]  # Örneklem (hız için)
    else:
        X_shap = X_test.iloc[:100].values
    
    # SHAP explainer
    if best_model_name == 'Logistic Regression':
        explainer = shap.LinearExplainer(best_model, X_train_scaled[:1000])
    else:
        explainer = shap.TreeExplainer(best_model)
    
    shap_values = explainer.shap_values(X_shap)
    
    # SHAP summary plot
    plt.figure(figsize=(10, 8))
    if isinstance(shap_values, list):
        shap_values_plot = shap_values[1]  # Pozitif sınıf için
    else:
        shap_values_plot = shap_values
    
    shap.summary_plot(
        shap_values_plot, X_shap,
        feature_names=X_test.columns[:len(shap_values_plot[0])],
        show=False, max_display=15
    )
    plt.title(f'SHAP Summary Plot - {best_model_name}')
    plt.tight_layout()
    plt.show()
    
    # Feature importance (ortalama SHAP değerleri)
    if isinstance(shap_values, list):
        shap_importance = np.abs(shap_values[1]).mean(0)
    else:
        shap_importance = np.abs(shap_values).mean(0)
    
    feature_importance_df = pd.DataFrame({
        'feature': X_test.columns[:len(shap_importance)],
        'importance': shap_importance
    }).sort_values('importance', ascending=False).head(15)
    
    print("\nEn Önemli 15 Özellik (SHAP):")
    print(feature_importance_df)
    
except ImportError:
    print("SHAP bulunamadı. 'pip install shap' ile yükleyebilirsiniz.")
except Exception as e:
    print(f"SHAP analizi sırasında hata: {e}")


In [ ]:
# Model ve Preprocessing Araçlarını Kaydetme

import pickle
import joblib
from pathlib import Path

# Model dosyalarını kaydetmek için klasör oluştur
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# Modelleri kaydet
for name, model in models.items():
    if name == 'Logistic Regression':
        # LR için scaler da kaydet
        joblib.dump(model, MODEL_DIR / f"{name.replace(' ', '_').lower()}.pkl")
        joblib.dump(scaler, MODEL_DIR / "scaler.pkl")
    else:
        joblib.dump(model, MODEL_DIR / f"{name.replace(' ', '_').lower()}.pkl")

# Preprocessing araçlarını kaydet
joblib.dump(num_imputer, MODEL_DIR / "num_imputer.pkl")
joblib.dump(cat_imputer, MODEL_DIR / "cat_imputer.pkl")

# Feature listelerini kaydet
feature_info = {
    'numeric_features': numeric_features_fe,
    'cat_features': cat_features,
    'feature_names': list(X.columns)
}

with open(MODEL_DIR / "feature_info.pkl", "wb") as f:
    pickle.dump(feature_info, f)

print("✅ Tüm modeller ve preprocessing araçları kaydedildi!")
print(f"📁 Kayıt yeri: {MODEL_DIR.absolute()}")


# 📋 Proje Özeti ve Sonuçlar

## ✅ Tamamlanan Adımlar:

1. ✅ **Veri Toplama ve Ön İnceleme**: Veri seti yüklendi ve genel yapı analiz edildi
2. ✅ **Veri Ön İşleme**: Eksik değerler dolduruldu, aykırı değerler temizlendi
3. ✅ **Feature Engineering**: Yeni özellikler oluşturuldu (KDA, verimlilik metrikleri)
4. ✅ **Modelleme**: 3 farklı model eğitildi (Logistic Regression, Random Forest, XGBoost)
5. ✅ **Kümeleme**: K-Means ile oyuncu persona analizi yapıldı
6. ✅ **Model Değerlendirme**: Tüm metrikler hesaplandı (Accuracy, Precision, Recall, F1, AUC-ROC)
7. ✅ **Açıklanabilir AI**: SHAP değerleri ile model açıklaması yapıldı
8. ✅ **Model Kaydetme**: Modeller ve preprocessing araçları kaydedildi
9. ✅ **Streamlit Arayüzü**: Kullanıcı dostu web arayüzü oluşturuldu

## 📊 Model Performans Özeti:

Yukarıdaki hücrelerde eğitilen modellerin performans metrikleri görüntülenebilir.

## 🚀 Kullanım:

1. **Notebook**: Tüm hücreleri sırayla çalıştırarak modelleri eğitin
2. **Streamlit**: `streamlit run app.py` komutu ile web arayüzünü başlatın
3. **Tahmin**: Streamlit arayüzünden oyuncu istatistiklerini girerek tahmin yapın

## 📁 Oluşturulan Dosyalar:

- `EDA.ipynb`: Ana analiz notebook'u
- `app.py`: Streamlit web arayüzü
- `models/`: Eğitilmiş modeller ve preprocessing araçları
- `requirements.txt`: Gerekli Python paketleri
- `README.md`: Proje dokümantasyonu

**Proje başarıyla tamamlandı! 🎉**
